In [2]:
#from mpl_toolkits import mplot3d
import array
import pandas
import matplotlib
import math   # 导入 math 模块
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pandas import Series, DataFrame
from sklearn.model_selection import train_test_split
from IPython.core.pylabtools import figsize
from sklearn.model_selection import cross_val_score
from sklearn import preprocessing
from sklearn.metrics import accuracy_score
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from scipy import stats
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold
from scipy.stats import pearsonr,spearmanr
from itertools import combinations
from scipy.special import comb, perm
from sklearn.kernel_ridge import KernelRidge
from sklearn.model_selection import ShuffleSplit
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import StandardScaler

rs = ShuffleSplit(n_splits=100, test_size=0.2, random_state=0)

scaler_M = MinMaxScaler()

data=pandas.read_csv(r"C:\Users\z\Desktop\模型统计性质分析\二元合金体积效应数据分类分析\Outliers\Select by Pearson cofficients from initial dataset.csv")

F=np.array(data)
p=data.ev_ratio
p=np.array(p)
F_0=F[:,3:67]

sf=np.array(data.sf)/100
Vsv=np.array(data.sv_Volume)
Vso=np.array(data.so_Volume)
N_sv=np.array(data.N_sv)
N_so=np.array(data.N_so)

In [4]:
a=pandas.read_csv(r"C:\Users\z\Desktop\图表及相应源代码\Fig 5\SBS AND SFS\KRR\hyper parameter container\lap\_alpha_SBS from initial-MA-clipped-17.csv")
b=pandas.read_csv(r"C:\Users\z\Desktop\图表及相应源代码\Fig 5\SBS AND SFS\KRR\hyper parameter container\lap\_gamma_SBS from initial-MA-clipped-17.csv")

_alpha=np.array(a).flatten()
_gamma=np.array(b).flatten()

In [5]:
evaluate_r_2=[]
evaluate_r_2_tr=[]

n=-1

for train_index, test_index in rs.split(F_0):
    
    scaler_M.fit(F_0[train_index,:])
    
    F_tr=scaler_M.transform(F_0[train_index,:])
    F_te=scaler_M.transform(F_0[test_index,:])
        
    n=n+1
    
    krr=KernelRidge(alpha=_alpha[n], 
                    #kernel='rbf',
                    kernel='laplacian',
                    gamma=_gamma[n])
    
    krr.fit(F_tr,(p.reshape(-1,1)[train_index,:]-1)*100)
    
    p_tr = krr.predict(F_tr)
    r_2_tr=r2_score((p.reshape(-1,1)[train_index,:]-1)*100, p_tr)
    evaluate_r_2_tr.append(r_2_tr)
    
    p_pre = krr.predict(F_te)   
    r_2=r2_score((p.reshape(-1,1)[test_index,:]-1)*100, p_pre)
    evaluate_r_2.append(r_2)

print(np.mean(evaluate_r_2_tr))
print(np.std(evaluate_r_2_tr))
print(np.mean(evaluate_r_2))
print(np.std(evaluate_r_2))

0.9998600807595469
0.0005887900577965783
0.2770986608257836
0.09673170843653128


In [6]:
std_r2=[]
ind=[]
r2=[]

std_r2_tr=[]
r2_tr=[]

u=np.shape(F_0)[1]-1
m=np.shape(F_0)[1]

In [8]:
sub=63

for l in range(0,sub):

    m_r_2=[]
    s_r_2=[]
    
    m_r_2_tr=[]
    s_r_2_tr=[]

    for k in range(0,np.shape(F_0)[1]):
        
        evaluate_r_2=[]
        evaluate_r_2_tr=[]
        
        n=-1
    
        F_d=np.delete(F_0,k,1)
    
        for train_index, test_index in rs.split(F_d):
            
            scaler_M.fit(F_d[train_index,:])
            F_tr=scaler_M.transform(F_d[train_index,:])
            F_te=scaler_M.transform(F_d[test_index,:])
        
            n=n+1
            
            krr=KernelRidge(alpha=_alpha[n], 
                            #kernel='rbf',
                            kernel='laplacian',
                            gamma=_gamma[n])
        
            krr.fit(F_tr,(p.reshape(-1,1)[train_index,:]-1)*100)
            
            p_tr = krr.predict(F_tr)
            r_2_tr=r2_score((p.reshape(-1,1)[train_index,:]-1)*100, p_tr)
            evaluate_r_2_tr.append(r_2_tr)
    
            p_pre = krr.predict(F_te)   
            r_2=r2_score((p.reshape(-1,1)[test_index,:]-1)*100, p_pre)
            evaluate_r_2.append(r_2)
        
        m_r_2.append(np.mean(evaluate_r_2))
        s_r_2.append(np.std(evaluate_r_2))
        
        m_r_2_tr.append(np.mean(evaluate_r_2_tr))
        s_r_2_tr.append(np.std(evaluate_r_2_tr))
    
    r2.append(max(m_r_2))
    t=m_r_2.index(max(m_r_2))
    std_r2.append(s_r_2[t])
    ind.append(t)
    
    r2_tr.append(m_r_2_tr[t])
    std_r2_tr.append(s_r_2_tr[t])
    
    F_0=np.delete(F_0,t,1)

In [10]:
#removing index
r=pd.DataFrame(ind)
r.to_csv('index.csv')

#training accuracy
R=pd.DataFrame(r2_tr)
R.to_csv('r2_tr.csv')
R=pd.DataFrame(std_r2_tr)
R.to_csv('std_r2_tr.csv')

#testing accuracy
r=pd.DataFrame(r2)
r.to_csv('r2.csv')
r=pd.DataFrame(std_r2)
r.to_csv('std_r2.csv')

In [5]:
evaluate_r_2=[]
evaluate_Ord=[]
evaluate_per=[]

#sf
evaluatesf_r_2=[]
evaluatesf_Ord=[]
evaluatesf_per=[]

In [6]:
n=-1

for train_index, test_index in rs.split(F_0):
    
    scaler_M.fit(F_0[train_index,:])
    
    F_tr=scaler_M.transform(F_0[train_index,:])
    F_te=scaler_M.transform(F_0[test_index,:])
      
    n=n+1
    
    krr=KernelRidge(alpha=_alpha[n], 
                    #kernel='rbf',
                    kernel='laplacian',
                    gamma=_gamma[n])
    
    krr.fit(F_tr,(p.reshape(-1,1)[train_index,:]-1)*100)
    p_pre = krr.predict(F_te)
    
    #VLF
    r_2=r2_score((p.reshape(-1,1)[test_index,:]-1)*100, p_pre) 
    result=pd.DataFrame(np.c_[(p.reshape(-1,1)[test_index,:]-1)*100,p_pre])
    corre_cof=result.corr('pearson')
    
    evaluate_per.append(corre_cof[0][1])
    evaluate_Ord.append(Ord)
    evaluate_r_2.append(r_2)
    
    #SF
    sf_pre=(((p_pre/100)+1)*Vso.reshape(-1,1)[test_index,:]-Vsv.reshape(-1,1)[test_index,:])/Vsv.reshape(-1,1)[test_index,:]
    r_2=r2_score(sf.reshape(-1,1)[test_index,:], sf_pre)
    result=pandas.DataFrame(np.c_[sf.reshape(-1,1)[test_index,:],sf_pre])
    corre_cof=result.corr('pearson')
    
    evaluatesf_per.append(corre_cof[0][1])
    evaluatesf_Ord.append(Ord)
    evaluatesf_r_2.append(r_2)

In [7]:
#VLF
m_per=np.mean(evaluate_per)
s_per=np.std(evaluate_per)
m_r_2=np.mean(evaluate_r_2)
s_r_2=np.std(evaluate_r_2)

print("VLF:")
print("r_2_mean=%.5f"%(m_r_2))
print("r_2_std=%.5f"%(s_r_2))
print("per_mean=%.5f"%(m_per))
print("per_std=%.5f"%(s_per))

#sf
m_RMSE=np.mean(evaluatesf_RMSE)
s_RMSE=np.std(evaluatesf_RMSE)
m_MAE=np.mean(evaluatesf_MAE)
s_MAE=np.std(evaluatesf_MAE)
m_r_2=np.mean(evaluatesf_r_2)
s_r_2=np.std(evaluatesf_r_2)
m_per=np.mean(evaluatesf_per)
s_per=np.std(evaluatesf_per)

print("SF:")
print("r_2_mean=%.5f"%(m_r_2))
print("r_2_std=%.5f"%(s_r_2))
print("per_mean=%.5f"%(m_per))
print("per_std=%.5f"%(s_per))

VLF:
r_2_mean=0.66596
r_2_std=0.07031
MAE_mean=5.83365
MAE_std=0.70146
RMSE_mean=8.73626
RMSE_std=1.12561
per_mean=0.82248
per_std=0.04355
Ord_mean=0.82195
Ord_std=0.02397
SF:
r_2_mean=0.79572
r_2_std=0.08467
MAE_mean=7.01829
MAE_std=1.07350
RMSE_mean=11.65030
RMSE_std=2.56126
per_mean=0.90234
per_std=0.03503
Ord_mean=0.89245
Ord_std=0.01559
